In [2]:
import pandas as pd
import geopandas as gpd

# 07번에서 저장했던 최종 통합 파일 불러오기
final_merged = pd.read_csv('grid_HSI_final_merged.csv')

print("불러오기 완료")
print("행 수:", len(final_merged))
print("컬럼 수:", len(final_merged.columns))

불러오기 완료
행 수: 11388
컬럼 수: 212


In [3]:
## 1단계: 큰 덩어리 제외 

import pandas as pd
import numpy as np

# final_merged: 212컬럼 최종 통합 파일 (이미 로드되어 있다고 가정)
print("시작 컬럼 수:", len(final_merged.columns))

# ── 1단계-A: 메타/키 컬럼 (8개) ──
meta_cols = [
    'grid_id', 'geometry',
    'ADM_SIDO', 'ADM_SGG_CD', 'ADM_EMD_CD', 'ADM_EMD_NM',
    'lon', 'lat',
]

# ── 1단계-B: 검증용 컬럼 (7개) ──
validation_cols = [
    'CLOUD_20240602', 'CLOUD_20240618', 'CLOUD_20240704',
    'CLOUD_20240805', 'CLOUD_20240922',
    'ET_202308_valid', 'LC_covered_ratio',
]

# ── 1단계-C: 인구 시간대 원본 (48개, work_00h~23h + visi_00h~23h) ──
hourly_pop_cols = (
    [f'work_{h:02d}h' for h in range(24)] +
    [f'visi_{h:02d}h' for h in range(24)]
)

# ── 1단계-D: FOREST_RATIO 계열 (24개, 1km 해상도 - 격자 간 변별력 없음) ──
forest_cols = [
    'FOREST_TYPE_CD', 'FOREST_TYPE_NM',
    'FOREST_RATIO_D', 'FOREST_RATIO_PD', 'FOREST_RATIO_PK', 'FOREST_RATIO_PL',
    'FOREST_RATIO_PR', 'FOREST_RATIO_PT', 'FOREST_RATIO_PC', 'FOREST_RATIO_PCr',
    'FOREST_RATIO_PCo', 'FOREST_RATIO_PCa', 'FOREST_RATIO_H', 'FOREST_RATIO_Q',
    'FOREST_RATIO_PH', 'FOREST_RATIO_B', 'FOREST_RATIO_M', 'FOREST_RATIO_O',
    'FOREST_RATIO_F', 'FOREST_RATIO_E', 'FOREST_RATIO_L', 'FOREST_RATIO_LP',
    'FOREST_RATIO_R', 'FOREST_RATIO_W',
]

stage1_drop = meta_cols + validation_cols + hourly_pop_cols + forest_cols

# 실제 존재하는 컬럼만 필터링 (혹시 이름 다른 경우 대비)
stage1_drop_existing = [c for c in stage1_drop if c in final_merged.columns]
missing_from_data = [c for c in stage1_drop if c not in final_merged.columns]

print(f"\n1단계 제외 대상: {len(stage1_drop)}개")
print(f"실제 존재해서 제외됨: {len(stage1_drop_existing)}개")
if missing_from_data:
    print(f"⚠️ 컬럼명이 안 맞아서 못 찾은 것: {missing_from_data}")

# 메타 컬럼은 나중에 필요하니 별도 보관 후 분석용 df에서만 제외
meta_keep = final_merged[[c for c in meta_cols if c in final_merged.columns]].copy()

df_stage1 = final_merged.drop(columns=stage1_drop_existing, errors='ignore')
print(f"\n1단계 완료 후 컬럼 수: {len(df_stage1.columns)} (목표: 125)")

시작 컬럼 수: 212

1단계 제외 대상: 87개
실제 존재해서 제외됨: 87개

1단계 완료 후 컬럼 수: 125 (목표: 125)


In [4]:
## 2단계: 축 배정 + 중복 제거
# ── 2단계-①: LC2 (20개) + LC3 (3개) 제외 ──
lc2_cols = [
    'LC2_110_주거지역_ratio', 'LC2_120_공업지역_ratio', 'LC2_130_상업지역_ratio',
    'LC2_140_문화·체육·휴양지역_ratio', 'LC2_150_교통지역_ratio', 'LC2_160_공공시설지역_ratio',
    'LC2_210_논_ratio', 'LC2_220_밭_ratio', 'LC2_230_시설재배지_ratio',
    'LC2_240_과수원_ratio', 'LC2_250_기타재배지_ratio',
    'LC2_310_활엽수림_ratio', 'LC2_320_침엽수림_ratio', 'LC2_330_혼효림_ratio',
    'LC2_410_자연초지_ratio', 'LC2_420_인공초지_ratio',
    'LC2_510_내륙습지_ratio',
    'LC2_610_자연나지_ratio', 'LC2_620_인공나지_ratio',
    'LC2_710_내륙수_ratio',
]
lc3_cols = ['LC3_dom_code', 'LC3_dom_name', 'LC3_dom_ratio']

# ── 2단계-①: LC1 내부 축소 (시가화건조지역_ratio만 남김, 나머지 9개 제외) ──
lc1_drop_cols = [
    'LC1_200_농업지역_ratio', 'LC1_300_산림지역_ratio', 'LC1_400_초지_ratio',
    'LC1_500_습지_ratio', 'LC1_600_나지_ratio', 'LC1_700_수역_ratio',
    'LC1_dom_code', 'LC1_dom_name', 'LC1_dom_ratio',
]
# 주의: LC1_300, LC1_400은 A축 녹지비율 파생변수 만들 때 잠깐 필요하니
#       바로 버리지 말고 파생변수 먼저 만든 뒤 제외

final_merged['GREEN_ratio_LC'] = (
    final_merged['LC1_300_산림지역_ratio'] + final_merged['LC1_400_초지_ratio']
)  # A축용 녹지비율 파생변수 (여기서 미리 생성)

# ── 2단계-②: LU 전체 제외 (10개, 신도시 단계로 이관) ──
lu_cols = [
    'LU_공공문화_ratio', 'LU_공업_ratio', 'LU_공원녹지_ratio', 'LU_교통_ratio',
    'LU_기반시설_ratio', 'LU_상업_ratio', 'LU_주거_ratio',
    'LU_dom_blockType', 'LU_zoneName', 'LU_plan_ratio',
]

# ── 2단계-④: ROAD 원본 전체 제외 (7개), 대신 평균차로수 파생변수 생성 ──
road_raw_cols = [
    'ROAD_len_m', 'ROAD_lane_len_m', 'ROAD_density_m_per_ha',
    'ROAD_max_speed', 'ROAD_top_rank', 'ROAD_n_links', 'ROAD_exist',
]

# 평균차로수 파생변수: 도로 있는 격자만 계산, 도로 없으면 NaN (표준화 단계에서 처리)
final_merged['ROAD_avg_lanes'] = np.where(
    final_merged['ROAD_len_m'] > 0,
    final_merged['ROAD_lane_len_m'] / final_merged['ROAD_len_m'],
    np.nan
)

stage2_drop = lc2_cols + lc3_cols + lc1_drop_cols + lu_cols + road_raw_cols
stage2_drop_existing = [c for c in stage2_drop if c in df_stage1.columns]
missing2 = [c for c in stage2_drop if c not in df_stage1.columns]

print(f"2단계 제외 대상: {len(stage2_drop)}개")
if missing2:
    print(f"⚠️ 컬럼명이 안 맞아서 못 찾은 것: {missing2}")

df_stage2 = df_stage1.drop(columns=stage2_drop_existing, errors='ignore')

# 파생변수 2개(GREEN_ratio_LC, ROAD_avg_lanes) 추가
df_stage2['GREEN_ratio_LC'] = final_merged['GREEN_ratio_LC']
df_stage2['ROAD_avg_lanes'] = final_merged['ROAD_avg_lanes']

print(f"\n2단계 완료 후 컬럼 수: {len(df_stage2.columns)} (목표: 77)")
print("\n최종 남은 컬럼 리스트:")
print(df_stage2.columns.tolist())

print(f"\nROAD_avg_lanes 결측 개수 (도로 없는 격자):", final_merged['ROAD_avg_lanes'].isna().sum())

2단계 제외 대상: 49개

2단계 완료 후 컬럼 수: 78 (목표: 77)

최종 남은 컬럼 리스트:
['m_10g_pop', 'm_20g_pop', 'm_30g_pop', 'm_40g_pop', 'm_50g_pop', 'm_60g_pop', 'w_10g_pop', 'w_20g_pop', 'w_30g_pop', 'w_40g_pop', 'w_50g_pop', 'w_60g_pop', 'wd_w_pop', 'wd_v_pop', 'we_w_pop', 'we_v_pop', 'pop_peak_h', 'pop_elderl', 'pop_lunch_', 'park_cnt', 'bus_cnt', 'subway_cnt', 'shade_cnt', 'senior_center_cnt', 'elder_class_cnt', 'elder_welfare_cnt', 'cooling_shelter_cnt', 'daycare_cnt', 'teen_fac_cnt', 'smart_shelter_cnt', 'tree_road_len', 'tree_road_cnt', 'NDVI_20240602', 'NDBI_20240602', 'NDWI_20240602', 'MNDWI_20240602', 'BSI_20240602', 'LST_C_20240602', 'NDVI_20240618', 'NDBI_20240618', 'NDWI_20240618', 'MNDWI_20240618', 'BSI_20240618', 'LST_C_20240618', 'NDVI_20240704', 'NDBI_20240704', 'NDWI_20240704', 'MNDWI_20240704', 'BSI_20240704', 'LST_C_20240704', 'NDVI_20240805', 'NDBI_20240805', 'NDWI_20240805', 'MNDWI_20240805', 'BSI_20240805', 'LST_C_20240805', 'NDVI_20240922', 'NDBI_20240922', 'NDWI_20240922', 'MNDWI_20240

In [5]:
## 3단계: 위성지수 시계열 압축 (다중공선성·결측 처리)

# ── 개별시기 원본 전체 제외 (5개 시기 x 6개 지수 = 30개) ──
scene_dates = ['20240602', '20240618', '20240704', '20240805', '20240922']
scene_indices = ['LST_C', 'NDVI', 'NDBI', 'NDWI', 'MNDWI', 'BSI']

individual_scene_cols = [f'{idx}_{date}' for idx in scene_indices for date in scene_dates]

# ── max_5scene 제외 (LST, NDVI만 존재, 2개) ──
max_5scene_cols = ['LST_C_max_5scene', 'NDVI_max_5scene']

stage3_drop = individual_scene_cols + max_5scene_cols
stage3_drop_existing = [c for c in stage3_drop if c in df_stage2.columns]
missing3 = [c for c in stage3_drop if c not in df_stage2.columns]

print(f"3단계 제외 대상: {len(stage3_drop)}개")
if missing3:
    print(f"⚠️ 컬럼명이 안 맞아서 못 찾은 것: {missing3}")

# ── NDBI/NDWI mean_5scene은 축엔 안 쓰되, 검증용으로 별도 보관 ──
verification_cols = ['NDBI_mean_5scene', 'NDWI_mean_5scene']
df_verification = df_stage2[[c for c in verification_cols if c in df_stage2.columns]].copy()

df_stage3 = df_stage2.drop(columns=stage3_drop_existing + verification_cols, errors='ignore')

print(f"\n3단계 완료 후 컬럼 수: {len(df_stage3.columns)} (목표: 46)")
print("\n최종 E/S/A 축 배정용 컬럼 리스트:")
print(df_stage3.columns.tolist())

print(f"\n검증용 보류 컬럼(별도 보관, df_verification): {df_verification.columns.tolist()}")

3단계 제외 대상: 32개

3단계 완료 후 컬럼 수: 44 (목표: 46)

최종 E/S/A 축 배정용 컬럼 리스트:
['m_10g_pop', 'm_20g_pop', 'm_30g_pop', 'm_40g_pop', 'm_50g_pop', 'm_60g_pop', 'w_10g_pop', 'w_20g_pop', 'w_30g_pop', 'w_40g_pop', 'w_50g_pop', 'w_60g_pop', 'wd_w_pop', 'wd_v_pop', 'we_w_pop', 'we_v_pop', 'pop_peak_h', 'pop_elderl', 'pop_lunch_', 'park_cnt', 'bus_cnt', 'subway_cnt', 'shade_cnt', 'senior_center_cnt', 'elder_class_cnt', 'elder_welfare_cnt', 'cooling_shelter_cnt', 'daycare_cnt', 'teen_fac_cnt', 'smart_shelter_cnt', 'tree_road_len', 'tree_road_cnt', 'NDVI_mean_5scene', 'LST_C_mean_5scene', 'SM_202307_mean', 'SM_202307_max', 'SM_202307_min', 'SM_202307_std', 'ET_202308_mean', 'ET_202308_max', 'ET_202308_min', 'LC1_100_시가화건조지역_ratio', 'GREEN_ratio_LC', 'ROAD_avg_lanes']

검증용 보류 컬럼(별도 보관, df_verification): ['NDBI_mean_5scene', 'NDWI_mean_5scene']


In [6]:
## 4단계: ET/SM 제외, 인구 원본 압축

# ── ET(증발산) 전체 제외 (3개) — 결측 75.77%, 기본산정 제외, 보고서 한계점으로만 언급 ──
et_cols = ['ET_202308_mean', 'ET_202308_max', 'ET_202308_min']

# ── SM(토양수분) 전체 제외 (4개) — 값 범위 0.017로 변별력 없음, 보고서 한계점으로만 언급 ──
sm_cols = ['SM_202307_mean', 'SM_202307_max', 'SM_202307_min', 'SM_202307_std']

# ── 인구 원본 전체 제외 (16개) — pop_peak_h/pop_elderl/pop_lunch_ 3개 파생변수로 대체 ──
pop_raw_cols = (
    [f'{g}_{age}g_pop' for g in ['m', 'w'] for age in [10, 20, 30, 40, 50, 60]] +
    ['wd_w_pop', 'wd_v_pop', 'we_w_pop', 'we_v_pop']
)

stage4_drop = et_cols + sm_cols + pop_raw_cols
stage4_drop_existing = [c for c in stage4_drop if c in df_stage3.columns]
missing4 = [c for c in stage4_drop if c not in df_stage3.columns]

print(f"4단계 제외 대상: {len(stage4_drop)}개")
if missing4:
    print(f"⚠️ 컬럼명이 안 맞아서 못 찾은 것: {missing4}")

df_stage4 = df_stage3.drop(columns=stage4_drop_existing, errors='ignore')

print(f"\n4단계 완료 후 컬럼 수: {len(df_stage4.columns)} (목표: 21)")
print("\n최종 E/S/A 축 배정용 컬럼 리스트:")
print(df_stage4.columns.tolist())

4단계 제외 대상: 23개

4단계 완료 후 컬럼 수: 21 (목표: 21)

최종 E/S/A 축 배정용 컬럼 리스트:
['pop_peak_h', 'pop_elderl', 'pop_lunch_', 'park_cnt', 'bus_cnt', 'subway_cnt', 'shade_cnt', 'senior_center_cnt', 'elder_class_cnt', 'elder_welfare_cnt', 'cooling_shelter_cnt', 'daycare_cnt', 'teen_fac_cnt', 'smart_shelter_cnt', 'tree_road_len', 'tree_road_cnt', 'NDVI_mean_5scene', 'LST_C_mean_5scene', 'LC1_100_시가화건조지역_ratio', 'GREEN_ratio_LC', 'ROAD_avg_lanes']


In [7]:
## 5단계: 중복·희소 지표 정리 (tree_road, 노인시설, 교통시설)

# ── 노인시설 3종 통합 -> 개별 컬럼 제외 전에 파생변수 먼저 생성 ──
df_stage4['senior_facility_total'] = (
    df_stage4['senior_center_cnt'] +
    df_stage4['elder_class_cnt'] +
    df_stage4['elder_welfare_cnt']
)

# ── 5단계 제외 목록 ──
stage5_drop = [
    'tree_road_cnt',                                   # tree_road_len과 중복(상관계수 0.93)
    'senior_center_cnt', 'elder_class_cnt', 'elder_welfare_cnt',  # senior_facility_total로 통합
    'bus_cnt', 'subway_cnt',                           # E/S/A축 미배정, 동선분석 전용
]

stage5_drop_existing = [c for c in stage5_drop if c in df_stage4.columns]
missing5 = [c for c in stage5_drop if c not in df_stage4.columns]

print(f"5단계 제외 대상: {len(stage5_drop)}개")
if missing5:
    print(f"⚠️ 컬럼명이 안 맞아서 못 찾은 것: {missing5}")

df_stage5 = df_stage4.drop(columns=stage5_drop_existing, errors='ignore')

print(f"\n5단계 완료 후 컬럼 수: {len(df_stage5.columns)} (목표: 16)")
print("\n최종 축 배정용 컬럼 리스트:")
print(df_stage5.columns.tolist())

5단계 제외 대상: 6개

5단계 완료 후 컬럼 수: 16 (목표: 16)

최종 축 배정용 컬럼 리스트:
['pop_peak_h', 'pop_elderl', 'pop_lunch_', 'park_cnt', 'shade_cnt', 'cooling_shelter_cnt', 'daycare_cnt', 'teen_fac_cnt', 'smart_shelter_cnt', 'tree_road_len', 'NDVI_mean_5scene', 'LST_C_mean_5scene', 'LC1_100_시가화건조지역_ratio', 'GREEN_ratio_LC', 'ROAD_avg_lanes', 'senior_facility_total']


In [8]:
## 5단계 추가: 청소년시설 제외 (취약계층 근거 불충분)

df_stage5 = df_stage5.drop(columns=['teen_fac_cnt'], errors='ignore')

print(f"최종 컬럼 수: {len(df_stage5.columns)} (목표: 15)")
print(df_stage5.columns.tolist())

최종 컬럼 수: 15 (목표: 15)
['pop_peak_h', 'pop_elderl', 'pop_lunch_', 'park_cnt', 'shade_cnt', 'cooling_shelter_cnt', 'daycare_cnt', 'smart_shelter_cnt', 'tree_road_len', 'NDVI_mean_5scene', 'LST_C_mean_5scene', 'LC1_100_시가화건조지역_ratio', 'GREEN_ratio_LC', 'ROAD_avg_lanes', 'senior_facility_total']


In [9]:
## 5단계 추가: ROAD_avg_lanes 제외 (결측 82%, 3단계 링크비용함수로 이관)

df_stage5 = df_stage5.drop(columns=['ROAD_avg_lanes'], errors='ignore')

print(f"최종 컬럼 수: {len(df_stage5.columns)} (목표: 14)")
print(df_stage5.columns.tolist())

최종 컬럼 수: 14 (목표: 14)
['pop_peak_h', 'pop_elderl', 'pop_lunch_', 'park_cnt', 'shade_cnt', 'cooling_shelter_cnt', 'daycare_cnt', 'smart_shelter_cnt', 'tree_road_len', 'NDVI_mean_5scene', 'LST_C_mean_5scene', 'LC1_100_시가화건조지역_ratio', 'GREEN_ratio_LC', 'senior_facility_total']


In [10]:
## 6단계: 보류 지표 별도 저장 (검증용 + 3단계 링크비용함수용)

reserved_cols = {
    'NDBI_mean_5scene': '검증용 - LC1_100과 대조',
    'NDWI_mean_5scene': '검증용 - LC1_100과 대조',
    'ROAD_len_m': '3단계 링크비용함수 - 도로폭 계산용',
    'ROAD_lane_len_m': '3단계 링크비용함수 - 도로폭 계산용',
    'bus_cnt': '동선분석용(08번 노트북에서 이미 활용)',
    'subway_cnt': '동선분석용(08번 노트북에서 이미 활용)',
}

df_reserved = final_merged[['grid_id', 'geometry'] + list(reserved_cols.keys())].copy()

df_reserved.to_csv('grid_HSI_reserved_indicators.csv', index=False, encoding='utf-8-sig')

print("저장 완료: grid_HSI_reserved_indicators.csv")
print(f"행 수: {len(df_reserved):,}개, 컬럼 수: {len(df_reserved.columns)}개")
print()
print("포함 컬럼 및 용도:")
for col, use in reserved_cols.items():
    print(f"  - {col}: {use}")

저장 완료: grid_HSI_reserved_indicators.csv
행 수: 11,388개, 컬럼 수: 8개

포함 컬럼 및 용도:
  - NDBI_mean_5scene: 검증용 - LC1_100과 대조
  - NDWI_mean_5scene: 검증용 - LC1_100과 대조
  - ROAD_len_m: 3단계 링크비용함수 - 도로폭 계산용
  - ROAD_lane_len_m: 3단계 링크비용함수 - 도로폭 계산용
  - bus_cnt: 동선분석용(08번 노트북에서 이미 활용)
  - subway_cnt: 동선분석용(08번 노트북에서 이미 활용)


In [12]:
## 7단계: 최종 선정 지표 저장 (14개 확정본) 

final_indicators = ['pop_peak_h', 'pop_elderl', 'pop_lunch_', 'park_cnt', 'shade_cnt',
                     'cooling_shelter_cnt', 'daycare_cnt', 'smart_shelter_cnt',
                     'tree_road_len', 'NDVI_mean_5scene', 'LST_C_mean_5scene',
                     'LC1_100_시가화건조지역_ratio', 'GREEN_ratio_LC', 'senior_facility_total']

# 1단계에서 따로 빼뒀던 meta_keep(grid_id, geometry 등)을 다시 합침
df_final_selected = pd.concat([meta_keep, df_stage5[final_indicators]], axis=1)

df_final_selected.to_csv('grid_HSI_selected_indicators.csv', index=False, encoding='utf-8-sig')

print("저장 완료: grid_HSI_selected_indicators.csv")
print(f"행 수: {len(df_final_selected):,}개, 컬럼 수: {len(df_final_selected.columns)}개")
print(df_final_selected.columns.tolist())

저장 완료: grid_HSI_selected_indicators.csv
행 수: 11,388개, 컬럼 수: 22개
['grid_id', 'geometry', 'ADM_SIDO', 'ADM_SGG_CD', 'ADM_EMD_CD', 'ADM_EMD_NM', 'lon', 'lat', 'pop_peak_h', 'pop_elderl', 'pop_lunch_', 'park_cnt', 'shade_cnt', 'cooling_shelter_cnt', 'daycare_cnt', 'smart_shelter_cnt', 'tree_road_len', 'NDVI_mean_5scene', 'LST_C_mean_5scene', 'LC1_100_시가화건조지역_ratio', 'GREEN_ratio_LC', 'senior_facility_total']


In [14]:
## LU(토지이용계획) 보류 지표 별도 저장

reserved_cols = {
    'NDBI_mean_5scene': '검증용 - LC1_100과 대조',
    'NDWI_mean_5scene': '검증용 - LC1_100과 대조',
    'ROAD_len_m': '3단계 링크비용함수 - 도로폭 계산용',
    'ROAD_lane_len_m': '3단계 링크비용함수 - 도로폭 계산용',
    'bus_cnt': '동선분석용(08번 노트북에서 이미 활용)',
    'subway_cnt': '동선분석용(08번 노트북에서 이미 활용)',
    'LU_공공문화_ratio': '5~6단계 신도시 적용 - 판교분당 LU-HSI 관계를 왕숙창릉 LU에 적용',
    'LU_공업_ratio': '5~6단계 신도시 적용',
    'LU_공원녹지_ratio': '5~6단계 신도시 적용',
    'LU_교통_ratio': '5~6단계 신도시 적용',
    'LU_기반시설_ratio': '5~6단계 신도시 적용',
    'LU_상업_ratio': '5~6단계 신도시 적용',
    'LU_주거_ratio': '5~6단계 신도시 적용',
    'LU_dom_blockType': '5~6단계 신도시 적용',
    'LU_zoneName': '5~6단계 신도시 적용',
    'LU_plan_ratio': '5~6단계 신도시 적용',
}

df_reserved = final_merged[['grid_id', 'geometry'] + list(reserved_cols.keys())].copy()

df_reserved.to_csv('grid_HSI_reserved_indicators.csv', index=False, encoding='utf-8-sig')

print("저장 완료: grid_HSI_reserved_indicators.csv")
print(f"행 수: {len(df_reserved):,}개, 컬럼 수: {len(df_reserved.columns)}개")
print()
print("포함 컬럼 및 용도:")
for col, use in reserved_cols.items():
    print(f"  - {col}: {use}")

저장 완료: grid_HSI_reserved_indicators.csv
행 수: 11,388개, 컬럼 수: 18개

포함 컬럼 및 용도:
  - NDBI_mean_5scene: 검증용 - LC1_100과 대조
  - NDWI_mean_5scene: 검증용 - LC1_100과 대조
  - ROAD_len_m: 3단계 링크비용함수 - 도로폭 계산용
  - ROAD_lane_len_m: 3단계 링크비용함수 - 도로폭 계산용
  - bus_cnt: 동선분석용(08번 노트북에서 이미 활용)
  - subway_cnt: 동선분석용(08번 노트북에서 이미 활용)
  - LU_공공문화_ratio: 5~6단계 신도시 적용 - 판교분당 LU-HSI 관계를 왕숙창릉 LU에 적용
  - LU_공업_ratio: 5~6단계 신도시 적용
  - LU_공원녹지_ratio: 5~6단계 신도시 적용
  - LU_교통_ratio: 5~6단계 신도시 적용
  - LU_기반시설_ratio: 5~6단계 신도시 적용
  - LU_상업_ratio: 5~6단계 신도시 적용
  - LU_주거_ratio: 5~6단계 신도시 적용
  - LU_dom_blockType: 5~6단계 신도시 적용
  - LU_zoneName: 5~6단계 신도시 적용
  - LU_plan_ratio: 5~6단계 신도시 적용
